# Member 1 — MobileNet Transfer Learning

**Model: Transfer Model 1 (MobileNetV3Small, ImageNet-pretrained) — owned by
Member 1 (group leader).** Binary malaria diagnosis: Parasitized (1, positive
class) vs Uninfected (0) on the NIH/NLM cell-image set (27,558 images).

This notebook is **my only workstream**: my model, my experiments, my
TensorBoard evidence, my evaluation, my explainability, my error analysis, and
my contribution to the group report. Members 2–4 build their own models in
their own folders.

**Targets to report and discuss** (assignment §5.1): transfer-learning models
aim at ≥95% sensitivity (recall on Parasitized) and ≥90% specificity (TNR on
Uninfected) on the held-out test set. These guide reporting — they are not
automatic pass/fail gates.

In [ ]:
# Section 2 — Imports and Configuration
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.environ["TB_LOGDIR"] = ("/content/drive/MyDrive/malaria_project/"
                               "logs/tensorboard/transfer_model_1")
    DRIVE_DATA = "/content/drive/MyDrive/malaria_project/data"
else:
    os.environ.setdefault("TB_LOGDIR", "logs/tensorboard/transfer_model_1")
    DRIVE_DATA = "data"

# Shared group utilities live in the repo root (common/, src/).
# On Colab: clone the repo once; locally the repo is already the cwd.
if not pathlib.Path("common").exists():
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/k-nizy/"
                    "Malaria-CNN-Classification-Project-.git"], check=True)
    for item in ("common", "src"):
        candidate = pathlib.Path("Malaria-CNN-Classification-Project-") / item
        if candidate.exists():
            candidate.rename(item)
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from common.data import (SEED, LABEL_MAP, get_data, count_images,
                         build_datasets, split_fingerprint,
                         make_synthetic_cell_dataset)
from common.models import build_transfer_model, unfreeze_top_n
from common.eval import (evaluate_model, evaluate_metrics, Metrics,
                         collect_predictions, predictions_from_probs,
                         plot_training_curves, plot_confusion_matrix,
                         plot_roc_curve, find_misclassifications,
                         plot_misclassifications)
from common.tb import ExperimentLogger, make_run_name, zip_logs_for_drive
from src.utils.project_config import get_model_config  # group registration

# --- configuration: single source of truth for every experiment -----------
SEED       = 42
IMG_SIZE   = (224, 224)    # MobileNetV3 minimum input size (Keras requirement)
BATCH_SIZE = 32
EPOCHS     = 30
RUN_PREFIX = "mobilenet"   # run names: mobilenet_exp_01_baseline (spec §8)

cfg = get_model_config("transfer_model_1")
print("Registered as:", cfg.owner, "->", cfg.architecture)
tf.keras.utils.set_random_seed(SEED)
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("TensorBoard logdir:", os.environ["TB_LOGDIR"])

## Section 3 — Dataset Inspection

The NIH/NLM malaria cell-image set ships as one zip (Parasitized/ +
Uninfected/ folders, 13,779 images each). `get_data` downloads it once and
caches it in Drive, so later Colab sessions skip the ~700 MB download.
Inspection below prints real counts and shows random cells from each class —
no invented statistics.

In [ ]:
# Section 3 — Dataset Inspection (the ONLY cell that touches the real dataset)
cell_images = get_data(drive_data_dir=DRIVE_DATA)
counts = count_images(cell_images)
print("per-class counts:", counts, "| total:", sum(counts.values()))

In [ ]:
# Sample grid with the clinical convention built in:
# class_names order => Uninfected=0, Parasitized=1 (matches training labels)
peek = tf.keras.utils.image_dataset_from_directory(
    str(cell_images), image_size=IMG_SIZE, batch_size=12,
    shuffle=True, seed=SEED, class_names=["Uninfected", "Parasitized"])

for x, y in peek.take(1):
    print("image shape:", x.shape[1:], "| labels:", y.numpy().ravel())
    fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
    for ax, img, lab in zip(axes.ravel(), x, y):
        ax.imshow(img.numpy().astype("uint8"))
        ax.set_title("Parasitized" if int(lab) == 1 else "Uninfected",
                     fontsize=9)
        ax.axis("off")
    fig.suptitle("Random cells from the raw dataset")
    fig.tight_layout()
    plt.show()

## Section 4 — Preprocessing

Two deliberate choices:

1. **Backbone preprocessing lives inside the model graph.** MobileNetV3's
   `preprocess_input` (a pass-through: V3 expects raw 0–255 RGB) is a model
   layer, so the data pipeline stays raw end-to-end — double-preprocessing
   bugs are structurally impossible.
2. **Augmentation is training-only** — horizontal/vertical flips, ±15%
   rotation and zoom, applied by `build_datasets(augment=True)` to the train
   split only. Validation and test are never augmented (verified in tests).

In [ ]:
# Show what augmentation actually does to a few cells (display only —
# training uses the identical stack inside build_datasets(augment=True)).
augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.15, seed=SEED),
    tf.keras.layers.RandomZoom(0.15, seed=SEED),
])
for x, y in peek.take(1):
    x_aug = augmenter(x[:4], training=True)
    fig, axes = plt.subplots(2, 4, figsize=(11, 5))
    for j in range(4):
        axes[0, j].imshow(x[j].numpy().astype("uint8")); axes[0, j].set_title("original", fontsize=9)
        axes[1, j].imshow(x_aug[j].numpy().astype("uint8")); axes[1, j].set_title("augmented", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## Section 5 — Train / Validation / Test Setup (leakage discipline)

`build_datasets(seed=42)` splits **by file** into an exact 70/15/15, flipping
labels to the clinical convention (Parasitized=1). The **fingerprints** prove
the split is byte-identical for every member of the group — the basis for a
valid model comparison in the report.

Leakage rules enforced throughout this notebook:

- each image appears in exactly one split (file-level split, fixed seed);
- augmentation touches the **train split only**;
- experiments 01–07 report **validation** metrics only;
- the test set is evaluated **once**, after final model selection (Section 19).

In [ ]:
# Section 5 — Train/Validation/Test Setup
train_ds, val_ds, test_ds = build_datasets(
    cell_images, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_fraction=0.30, seed=SEED, augment=False)

print("train fingerprint:", split_fingerprint(train_ds))
print("val   fingerprint:", split_fingerprint(val_ds))
print("test  fingerprint:", split_fingerprint(test_ds))

## Section 6 — Evaluation Metrics

Clinical convention (shared with `src/utils/project_config.py`):

- **Parasitized = 1 = positive class**; Uninfected = 0
- **Sensitivity = recall on the Parasitized class** (infected cells found)
- **Specificity = true-negative rate on Uninfected** — a *different* quantity
  from recall; verified against a hand-built confusion case below before any
  training happens, so the convention is provably consistent.

Every experiment reports accuracy, precision, recall, F1, ROC-AUC (computed on
**validation**); sensitivity/specificity are the same numbers under their WHO
names and are reported in every row of the comparison table (§17).

In [ ]:
# Section 6 — metric convention sanity check (hand-built case)
# y_true=[1,1,0,0], all predictions correct => sensitivity=1 AND specificity=1
m_check = evaluate_metrics(y_true=np.array([1, 1, 0, 0]),
                           y_prob=np.array([0.9, 0.8, 0.2, 0.3]))
assert m_check.sensitivity == 1.0, "sensitivity must be recall on Parasitized"
assert m_check.specificity == 1.0, "specificity must be TNR on Uninfected"
# an always-negative classifier must show sensitivity=0 but specificity=1
m_neg = evaluate_metrics(y_true=np.array([1, 1, 0, 0]),
                         y_prob=np.array([0.1, 0.2, 0.1, 0.2]))
assert m_neg.sensitivity == 0.0 and m_neg.specificity == 1.0
print("convention verified:", m_check.format())

## Section 7 — TensorBoard Configuration

TensorBoard is the experiment evidence trail (mandatory): every run gets a
directory under `logs/tensorboard/transfer_model_1/` named exactly
`mobilenet_exp_XX_<description>` and carrying an HPARAMS record (lr, optimizer,
batch size, epochs, augmentation, dropout, L2, frozen/unfrozen strategy),
epoch scalars, a CSV log, and the end-of-run metrics **in the same directory**.
Raw event logs are preserved (never overwritten — each experiment gets a new
directory) and zipped to Drive for the report appendix.

In [ ]:
# Section 7 — TensorBoard setup
print("naming convention examples:")
for _i, _d in [(1, "baseline"), (5, "unfreeze_last20")]:
    print("  ", make_run_name(RUN_PREFIX, _i, _d))

# On Colab, uncomment to watch runs live:
# %load_ext tensorboard
# %tensorboard --logdir /content/drive/MyDrive/malaria_project/logs/tensorboard/transfer_model_1

# At the very end of the session, export the raw logs for the report appendix:
# zip_logs_for_drive(os.environ["TB_LOGDIR"])

## Section 8 — MobileNetV3Small Architecture (transfer learning)

```
Input (224x224x3, raw 0-255 RGB)
  -> MobileNetV3 preprocess (pass-through, inside the graph)
  -> MobileNetV3Small backbone (ImageNet, 939,697 params, frozen first)
  -> GlobalAveragePooling2D
  -> Dropout(0.3)
  -> Dense(1, sigmoid)  => P(Parasitized)
```

Why this backbone: **smallest** ImageNet model in Keras (~0.94 M params
without top), so each experiment trains fast and the 7+ run campaign stays
practical; MobileNetV3's squeeze-and-excitation blocks are well suited to
texture differences (parasite stains); and being clearly different from the
group's other transfer model strengthens the comparison. Keras requires a
minimum input of 224x224 for V3 — hence IMG_SIZE.

Strategy: **Phase 1** (experiments 01-04) trains only the head on frozen
features; **Phase 2** (05-07) unfreezes the top-N backbone layers with a 10x
lower learning rate (discriminative fine-tuning).

In [ ]:
# Section 8 — build the architecture + frozen-phase proof
def make_model(spec: dict) -> tf.keras.Model:
    # one experiment variant; `unfreeze_last` in spec switches to Phase 2
    model = build_transfer_model(
        "mobilenetv3small", img_size=IMG_SIZE,
        dropout=spec.get("dropout", 0.3),
        freeze_backbone="unfreeze_last" not in spec,
        l2=spec.get("l2", 0.0))
    if "unfreeze_last" in spec:
        unfreeze_top_n(model, n=spec["unfreeze_last"])   # BEFORE compile
    return model

model = make_model(dict(lr=1e-3))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
n_total = model.count_params()
n_train = int(sum(np.prod(w.shape) for w in model.trainable_weights))
print(f"{model.name}: {n_total:,} params total | trainable (frozen phase): {n_train:,}")
assert not model._malaria_backbone.trainable, "exp 01-04 require a frozen backbone"
assert any("preprocess" in l.name for l in model.layers), "preprocess must be in-graph"
model.summary()

## Section 9 — Experiment 01: `mobilenet_exp_01_baseline`

**Purpose:** establish the reference point — deliberately simple, defensible,
not optimized. **Single rule of the campaign:** every experiment changes ONE
clear variable against this baseline (or against the best evidence so far in
Phase 2), and the hypothesis is written *before* the run.

| field | value |
|---|---|
| architecture | MobileNetV3Small (ImageNet), frozen backbone |
| head | GAP -> Dropout(0.3) -> Dense(1, sigmoid) |
| optimizer / lr | Adam / 1e-3 |
| batch size / epochs | 32 / 30 |
| augmentation | none |
| regularization | dropout 0.3 only |
| metric basis | **validation** set (test stays untouched, §19) |

In [ ]:
# Section 9 — the experiment runner (used by ALL experiments, nothing drifts)
RUNS = []

def run_experiment(exp_id, name, spec, epochs=EPOCHS, verbose=2):
    exp = ExperimentLogger(
        model_name=RUN_PREFIX, exp_id=exp_id, description=name,
        hparams=dict(lr=spec["lr"], optimizer="adam", batch_size=BATCH_SIZE,
                     epochs=epochs, augmentation=spec.get("augment", False),
                     dropout=spec.get("dropout", 0.3), l2=spec.get("l2", 0.0),
                     unfreeze_last=spec.get("unfreeze_last", 0),
                     strategy=("frozen_head" if "unfreeze_last" not in spec
                               else f"unfreeze_last{spec['unfreeze_last']}"),
                     img_size=str(IMG_SIZE), seed=SEED,
                     model="MobileNetV3Small"),
        logdir=os.environ["TB_LOGDIR"])
    train_in = train_ds
    if spec.get("augment", False):
        # rebuild the train split with train-only augmentation (§5 rules)
        train_in, _, _ = build_datasets(cell_images, img_size=IMG_SIZE,
                                        batch_size=BATCH_SIZE, seed=SEED,
                                        augment=True)
    mdl = make_model(spec)
    mdl.compile(optimizer=tf.keras.optimizers.Adam(spec["lr"]),
                loss="binary_crossentropy", metrics=["accuracy"])
    history = mdl.fit(train_in, validation_data=val_ds, epochs=epochs,
                      callbacks=exp.callbacks(
                          checkpoint_path=str(exp.log_dir / "best.weights.h5")),
                      verbose=verbose)
    # VALIDATION metrics only — the test set stays untouched until §19
    val_metrics, _, _ = evaluate_model(mdl, val_ds)
    with tf.summary.create_file_writer(str(exp.log_dir)).as_default():
        for k, v in val_metrics.as_row().items():
            tf.summary.scalar(f"final_val/{k}", float(v), step=0)
    (exp.log_dir / "val_metrics.txt").write_text(
        "\n".join(f"{k}: {v}" for k, v in val_metrics.as_row().items()),
        encoding="utf-8")
    exp.log_hparams_once(val_metrics.as_row())
    print(f"[{exp.run_name}] {val_metrics.format()}")
    row = dict(run_name=exp.run_name, exp_id=exp_id, name=name, spec=spec,
               history=history, val_metrics=val_metrics, log_dir=exp.log_dir)
    RUNS.append(row)
    return row

# ---- Experiment 01 — baseline ----
exp01 = run_experiment(1, "baseline", dict(
    lr=1e-3, dropout=0.3, augment=False,
    note="Reference: frozen MobileNetV3Small, Adam 1e-3, bs 32, 30 epochs"))

## Section 10 — Experiment 02: `mobilenet_exp_02_lr_1e-4`

**Hypothesis (written before the run):** 1e-3 may overshoot on a small head
(Keras' Adam default is aggressive for 65k head parameters); 1e-4 should give
gentler convergence and possibly better generalization. **Single variable
vs EXP01:** learning rate 1e-3 → 1e-4.

In [ ]:
# Section 10 — Experiment 02
exp02 = run_experiment(2, "lr_1e-4", dict(
    lr=1e-4, dropout=0.3, augment=False,
    note="Single variable vs exp01: head LR 1e-3 -> 1e-4"))

## Section 11 — Experiment 03: `mobilenet_exp_03_augmentation`

**Hypothesis:** flip/rotation/zoom should widen the effective training set and
shrink the train/validation gap seen in the baseline curves. **Single variable
vs EXP01:** augmentation on (LR back to 1e-3 so the augmentation effect stays
separable from the LR effect).

In [ ]:
# Section 11 — Experiment 03
exp03 = run_experiment(3, "augmentation", dict(
    lr=1e-3, dropout=0.3, augment=True,
    note="Single variable vs exp01: train-only flip/rot/zoom augmentation"))

## Section 12 — Experiment 04: `mobilenet_exp_04_dropout_0.5`

**Hypothesis:** if the baseline still shows residual overfitting after
augmentation, stronger head regularization should close more of the gap.
**Single variable vs EXP01:** dropout 0.3 → 0.5 (no augmentation, so the
dropout effect is measured cleanly).

In [ ]:
# Section 12 — Experiment 04
exp04 = run_experiment(4, "dropout_0.5", dict(
    lr=1e-3, dropout=0.5, augment=False,
    note="Single variable vs exp01: head dropout 0.3 -> 0.5"))

## Section 13 — Experiment 05: `mobilenet_exp_05_unfreeze_last20`

**Hypothesis (Phase 2 begins):** the frozen ImageNet features were learned on
ImageNet classes, not malaria cells; letting the top-20 layers adapt should
raise sensitivity at a modest overfitting risk. **Variable vs best head
experiment:** unfreeze_last=20 at **1e-4** — the LR level EXP02/03 evidence
favours, 10x below the head phase (protects pretrained weights).

In [ ]:
# Section 13 — Experiment 05
exp05 = run_experiment(5, "unfreeze_last20", dict(
    lr=1e-4, dropout=0.3, augment=False, unfreeze_last=20,
    note="Phase 2: top-20 backbone layers trainable @ 1e-4 (discriminative FT)"))

## Section 14 — Experiment 06: `mobilenet_exp_06_finetune_lr_1e-5`

**Hypothesis:** fine-tuning at 1e-4 may still disturb pretrained features
(catastrophic forgetting shows as val curves that spike); 1e-5 should adapt
more slowly and stably. **Variable vs EXP05:** fine-tune LR 1e-4 → 1e-5,
same unfreeze depth.

In [ ]:
# Section 14 — Experiment 06
exp06 = run_experiment(6, "finetune_lr_1e-5", dict(
    lr=1e-5, dropout=0.3, augment=False, unfreeze_last=20,
    note="Variable vs exp05: fine-tune LR 1e-4 -> 1e-5, same depth"))

## Section 15 — Experiment 07: `mobilenet_exp_07_final_refinement`

**Purpose:** the best-configuration refinement — the recipe the evidence
actually supports. **[UPDATE THE SPEC BELOW FROM YOUR EXP01–06 RESULTS BEFORE
RUNNING]** — do not run this on defaults blindly. The default below assumes
the common outcome (fine-tuning + augmentation helps, stable LR wins); replace
it with what YOUR curves showed.

In [ ]:
# Section 15 — Experiment 07
# [FILL: set every field from the actual EXP01-06 evidence, keep the note honest]
exp07 = run_experiment(7, "final_refinement", dict(
    lr=1e-5, dropout=0.3, augment=True, unfreeze_last=20,
    note="Best-of refinement - [UPDATE spec + note from EXP01-06 evidence]"))

## Section 16 — Additional Experiments (only if justified)

Rule: run EXP08+ **only** when EXP01–07 evidence genuinely demands it — e.g.
EXP05 overfits badly → try a shallower `unfreeze_last=10`; EXP05 barely
changes anything → try deeper `unfreeze_last=40` at 1e-5. Never force extra
runs, and never delete poor results — failed experiments are documented
evidence of progression (the report discusses them).

```python
# Example (uncomment + adapt ONLY if justified):
# exp08 = run_experiment(8, "unfreeze_last40", dict(
#     lr=1e-5, dropout=0.3, augment=True, unfreeze_last=40,
#     note="Justification: [FILL from EXP05/06 evidence]"))
```

In [ ]:
# Section 17 — Experiment Comparison Table (from logged runs, not memory)
def comparison_table(runs) -> pd.DataFrame:
    rows = []
    for r in runs:
        m = r["val_metrics"].as_row()
        s = r["spec"]
        strategy = ("frozen_head" if "unfreeze_last" not in s
                    else f"unfreeze_last{s['unfreeze_last']}")
        config = strategy
        if s.get("augment"):
            config += " + augmentation"
        rows.append({
            "Experiment": r["run_name"],
            "Configuration": config,
            "Learning Rate": s["lr"],
            "Optimizer": "adam",
            "Batch Size": BATCH_SIZE,
            "Augmentation": bool(s.get("augment", False)),
            "Dropout/L2": f"{s.get('dropout', 0.3)}/{s.get('l2', 0.0)}",
            "Frozen/Unfrozen Strategy": strategy,
            "Accuracy": m["accuracy"],
            "Precision": m["precision"],
            "Recall": m["recall"],
            "F1": m["f1"],
            "ROC-AUC": m["roc_auc"],
            "Sensitivity": m["sensitivity"],
            "Specificity": m["specificity"],
            "Outcome/Observation": s.get("note", ""),
        })
    return pd.DataFrame(rows)

comparison_table(RUNS)

def scan_run_metrics(logdir=None) -> pd.DataFrame:
    # Post-restart recovery: rebuild metric rows from the run folders on disk
    # (each run wrote val_metrics.txt). Configuration lives in the run's
    # TensorBoard hparams + this notebook's cells.
    from pathlib import Path
    base = Path(logdir or os.environ["TB_LOGDIR"])
    rows = []
    for d in sorted(base.glob(f"{RUN_PREFIX}_exp_*")):
        f = d / "val_metrics.txt"
        if f.exists():
            pairs = [ln.split(": ", 1) for ln in f.read_text().splitlines()]
            rows.append(dict(Experiment=d.name,
                             **{k: float(v) for k, v in pairs}))
    return pd.DataFrame(rows)

print(scan_run_metrics())

In [ ]:
# Section 18 — Final Model Selection (multi-metric, validation-only evidence)
# Clinical priority first: sensitivity, then ROC-AUC (threshold-free), then
# specificity. Accuracy alone is NOT the criterion.
ranked = sorted(RUNS, key=lambda r: (r["val_metrics"].sensitivity,
                                     r["val_metrics"].roc_auc,
                                     r["val_metrics"].specificity),
                reverse=True)
print("Ranking (validation metrics only):")
for i, r in enumerate(ranked, 1):
    m = r["val_metrics"]
    print(f"{i}. {r['run_name']:<44} sens={m.sensitivity:.4f} "
          f"spec={m.specificity:.4f} auc={m.roc_auc:.4f} f1={m.f1:.4f}")

best = ranked[0]
print()
print(f"Selected: {best['run_name']}")
print("Rationale [FILL after real runs]: e.g. highest validation sensitivity "
      "with specificity at/above target and the best ROC-AUC; state the "
      "trade-off you accepted and why it is clinically defensible.")

# Rebuild the selected configuration and load its best checkpoint
final_spec = best["spec"]
final_model = make_model(final_spec)
final_model.compile(optimizer=tf.keras.optimizers.Adam(final_spec["lr"]),
                    loss="binary_crossentropy", metrics=["accuracy"])
final_model.load_weights(str(best["log_dir"] / "best.weights.h5"))
final_history = best["history"]

# confirm the checkpoint reproduces the selection evidence
check_metrics, _, _ = evaluate_model(final_model, val_ds)
print("checkpoint reproduction on validation:", check_metrics.format())

In [ ]:
# Section 19 — Final Test Evaluation (executed ONCE, after selection)
test_metrics, y_true_test, y_prob_test = evaluate_model(final_model, test_ds)
print("[FINAL TEST]", test_metrics.format())
print(f"targets: sensitivity >= 0.95 (got {test_metrics.sensitivity:.4f}), "
      f"specificity >= 0.90 (got {test_metrics.specificity:.4f})")

# write the final TEST metrics into the selected run's TensorBoard directory
final_logger = ExperimentLogger(model_name=RUN_PREFIX, exp_id=best["exp_id"],
                                description=best["name"],
                                logdir=os.environ["TB_LOGDIR"])
final_logger.log_final_metrics(test_metrics.as_row())

import json
from pathlib import Path
out = Path("results/metrics") / f"{final_logger.run_name}_final_test.json"
out.write_text(json.dumps(test_metrics.as_row(), indent=2), encoding="utf-8")
print("saved:", out)

In [ ]:
# Section 20 — Accuracy/Loss curves of the FINAL model
fig = plot_training_curves(final_history)
fig.savefig("results/figures/mobilenet_final_acc_loss_curves.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("[FILL interpretation: where does validation diverge from training? "
      "which epoch plateaus? connect to the experiments that addressed it]")

In [ ]:
# Section 21 — Confusion Matrix of the FINAL model
y_pred_test = predictions_from_probs(y_prob_test, 0.5)
fig = plot_confusion_matrix(y_true_test, y_pred_test)
fig.savefig("results/figures/mobilenet_final_confusion_matrix.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("[FILL interpretation: false negatives (infected->uninfected) are the "
       "clinically costly cell — state the counts explicitly]")

In [ ]:
# Section 22 — ROC/AUC curve of the FINAL model
fig = plot_roc_curve(y_true_test, y_prob_test)
fig.savefig("results/figures/mobilenet_final_roc.png",
            dpi=200, bbox_inches="tight")
plt.show()
print("[FILL interpretation: what does moving the threshold trade between "
       "sensitivity and specificity, and where would a screening system sit?]")

## Section 23 — Error Analysis

Real predictions only. For each error: **observation** (what the model saw/
predicted/confident) vs **possible explanation** (ambiguous cell, stain
variation, background artifact, image quality) — never a claimed medical
diagnosis. The curve diagnosis below ties overfitting/underfitting symptoms to
the specific experiments that responded to them.

In [ ]:
# Section 23 — misclassified examples from the FINAL model's test predictions
imgs, trues, preds, probs = find_misclassifications(final_model, test_ds, n=3)
fig = plot_misclassifications(imgs, trues, preds, probs)
fig.savefig("results/error_analysis/mobilenet_misclassified.png",
            dpi=200, bbox_inches="tight")
plt.show()

def diagnose_fit(history):
    h = history.history
    best_epoch = int(np.argmin(h["val_loss"])) + 1
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    notes = []
    if gap > 0.05:
        notes.append(f"OVERFITTING sign: train-val accuracy gap = {gap:.3f} "
                     "(response: exp03/exp04)")
    if min(h["val_loss"]) > 0.693:
        notes.append("UNDERFITTING sign: val_loss never beat the 0.693 chance line")
    if best_epoch < len(h["val_loss"]):
        notes.append(f"best val epoch = {best_epoch} of {len(h['val_loss'])}")
    return "; ".join(notes) or "no strong over/underfitting signal"

print("final-model curve diagnosis:", diagnose_fit(final_history))
print("[FILL: per-error observation vs possible explanation, in your own words]")

## Section 24 — Grad-CAM / Explainability

Grad-CAM is computed through the backbone's **last convolutional layer**
(identified by name below) using an eager `GradientTape` on the **logit** —
the sigmoid saturates on confident predictions and kills gradients otherwise
(verified behavior; see common/gradcam.py). It shows **where** the model
looked, never **whether** that looking is medically valid — interpretations
stay critical. Required example types: correct-infected, correct-uninfected,
and incorrect (when present in the test stream).

In [ ]:
# Section 24 — Grad-CAM on the FINAL model
from common.gradcam import (make_gradcam_heatmap, overlay_heatmap,
                            pick_gradcam_examples, _find_last_conv_layer,
                            gradcam_report_grid)

last_conv = _find_last_conv_layer(final_model)
print("Grad-CAM tap layer:", last_conv.name)

# gather the test stream (labels, probabilities, uint8 images) in one pass
ys, ps, ims = [], [], []
for x, y in test_ds:
    ys.append(y.numpy().ravel())
    ps.append(final_model.predict(x, verbose=0).ravel())
    ims.append(x.numpy().astype(np.uint8))
y_true_all = np.concatenate(ys)
y_prob_all = np.concatenate(ps)
images_all = np.concatenate(ims)

examples = pick_gradcam_examples(y_true_all, y_prob_all, images_all)
print("picked example types:", len(examples), "of 3 (incorrect included when present)")

# gradient-flow verification: a zero heatmap means the tap is broken
heat0, _ = make_gradcam_heatmap(examples[0][0], final_model)
assert heat0.max() > 0, "gradient flow broken - heatmap is empty"
print(f"gradient-flow check OK (heatmap max={heat0.max():.3f})")

fig = gradcam_report_grid(examples, final_model,
                          save_path="results/gradcam/mobilenet_gradcam.png")
plt.show()
print("[FILL: per-example interpretation - cell vs background activation, "
       "what the incorrect case reveals, shortcut concerns - your own words]")

## Section 25 — Final Findings

**[FILL — rewrite in your own voice before submission. Shape:]**

1. What the frozen features achieved (exp01) and what each controlled change
   actually bought or cost (exp02–04), in one sentence per experiment.
2. What fine-tuning changed and at what price (exp05–07), citing run names.
3. Final test result vs the ≥95% / ≥90% targets — state honestly.
4. The one error pattern that worried you most, and what (if anything)
   addresses it.
5. What Grad-CAM showed and what it cannot prove.

### Submission audit (spec §25)

| item | evidence |
|---|---|
| pretrained transfer-learning model | §8 build + frozen-phase assert |
| only Member 1 files touched | this notebook + `experiments/transfer_model_1/` |
| no train/test leakage | §5 fingerprints; test evaluated once (§19) |
| baseline exists | `mobilenet_exp_01_baseline` (§9) |
| ≥7 meaningful, non-duplicate experiments | §9–15 (+ §16 if any) |
| TensorBoard for every experiment | `logs/tensorboard/transfer_model_1/<run>/` |
| descriptive run names | `make_run_name` output in §7 |
| acc/precision/recall/F1/ROC-AUC per experiment | §17 table |
| final sensitivity + specificity | §19 FINAL TEST print |
| final model selected on evidence | §18 ranking + rationale |
| 4 required figures | §20–22 (+ single-row report figure below) |
| Grad-CAM: 3 example types + gradient check | §24 |
| error analysis + over/underfitting | §23 |
| no fake results / citations / dead code | audit yourself before submitting |

In [ ]:
# Bonus artifact for the report: the four required plots in ONE row
from common.eval import final_evaluation_figure
fig_row, m_row, _, _ = final_evaluation_figure(
    final_model, final_history, test_ds,
    save_path="results/figures/mobilenet_final_evaluation_row.png")
plt.show()
print(m_row.format())